In [1]:
%load_ext autoreload
%autoreload 2

In [15]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.io import load_pkg_yaml
from icare_risk.utils.paths import get_path

DATA_DIR = get_path('data/synthetic/1000p')
DATA_PROC_DIR = DATA_DIR / 'processed'
filename = '03_derived_analysis_dataset.parquet'
df = pd.read_parquet(DATA_PROC_DIR / filename)


In [16]:
print(df.columns.tolist())

['SUBJECT', 'SPELL_IDENTIFIER', 'ENCNTR_ID', 'EPISODE_IDENTIFIER', 'ADMISSION_DATE', 'ADMISSION_TIME', 'AGE_AT_ADMISSION', 'DISCHARGE_DATE', 'MAIN_SPECIALTY_CODE', 'MAIN_SPECIALTY_CODE_DESC', 'INDEX_OF_MULTIPLE_DEPRIVATION_DECILE', 'index_admission', 'definitions', 'charlson_hx_mi', 'charlson_hx_chf', 'charlson_hx_pvd', 'charlson_hx_stroke', 'charlson_hx_dementia', 'charlson_hx_pulmonary', 'charlson_hx_rheum', 'charlson_hx_pud', 'charlson_hx_liver_mild', 'charlson_hx_diabetes_uncomp', 'charlson_hx_hemiplegia', 'charlson_hx_renal_mod_sev', 'charlson_hx_diabetes_comp', 'charlson_hx_cancer_solid', 'charlson_hx_leukemia', 'charlson_hx_lymphoma', 'charlson_hx_liver_mod_sev', 'charlson_hx_cancer_met', 'charlson_hx_aids', 'charlson_hx_hiv', 'kim_hx_prior_esbl', 'kim_hx_recent_hosp_1yr', 'kim_hx_nursing_home_resident', 'kim_hx_urinary_catheter', 'kim_hx_prior_antibiotics_90d', 'kim_hx_recent_antibiotics_90d', 'tumbarello_hx_prior_esbl', 'tumbarello_hx_recent_hosp_90d', 'tumbarello_hx_recent_ab

In [17]:
# Check which columns have 100% missing values
empty_cols = df.columns[df.isna().all()]
print("Columns with all NaNs:")
for c in empty_cols:
    print(f"  - {c}")

Columns with all NaNs:
  - charlson_hx_diabetes_comp_dslast
  - charlson_hx_diabetes_comp_dsfirst
  - charlson_hx_leukemia_dslast
  - charlson_hx_leukemia_dsfirst
  - days_since_last_discharge
  - qualifying_last_abx


In [ ]:
# --------------------------------------------------------------------
# Display TableOne characteristics
# --------------------------------------------------------------------
# Libraries
from pathlib import Path
import numpy as np
import pandas as pd
from tableone import TableOne

# Comment to see warnings
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# 1. Load your master dataset
DATA_DIR = Path("/app/data/synthetic/1000p/processed/")
filename = '03_derived_analysis_dataset.parquet'
df_master = pd.read_parquet(DATA_DIR / filename)

# 1. Ensure auxiliary mockup columns exist if not already present
if "esbl_label" not in df_master.columns:
    df_master["esbl_label"] = np.random.choice(["Positive", "Negative"], size=len(df_master), p=[0.15, 0.85])


import pandas as pd

def convert_dtypes_and_classify(
    df,
    force_categorical=None,
    force_numerical=None
):
    """
    Explicitly casts columns and partitions them into categorical vs numerical features
    for Table 1, with optional manual override lists.
    """
    df = df.copy()

    # Default empty sets/lists if none provided
    force_categorical = set(force_categorical or [])
    force_numerical = set(force_numerical or [])

    # Define columns to exclude from the statistical summary table
    exclude_cols = {
        "SUBJECT", "SPELL_IDENTIFIER", "ENCNTR_ID", "EPISODE_IDENTIFIER",
        "ADMISSION_DATE", "ADMISSION_TIME", "DISCHARGE_DATE", "DEATH_DATE",
        "BIRTH_DATE", "UPDATE_DT_TM", "index_admission",
        "MAIN_SPECIALTY_CODE", "GENDER", "ETHNICITY", "definitions",
        "closest_culture_date", "esbl_status", "esbl_label"
    }

    # Candidate features for summary
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    categorical_cols = []
    numerical_cols = []

    for col in feature_cols:
        # --- 1. Manual Overrides Take Absolute Priority ---
        if col in force_categorical:
            categorical_cols.append(col)
            df[col] = df[col].fillna("Missing").astype(str)
            continue

        if col in force_numerical:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            numerical_cols.append(col)
            continue

        # --- 2. Automatic Rules ---
        is_object_or_cat = df[col].dtype == "object" or df[col].dtype.name == "category"
        is_binary_flag = col.startswith("hx_") and not col.endswith(("dslast", "dsfirst", "_cnt"))
        is_low_cardinality_int = df[col].nunique() <= 3 and set(df[col].dropna().unique()).issubset({0, 1, 0.0, 1.0})
        is_known_desc = col.endswith("_DESC") or col.endswith("_DES") or col in {"sex", "gender", "ethnicity"}

        if is_object_or_cat or is_binary_flag or is_low_cardinality_int or is_known_desc:
            categorical_cols.append(col)
            df[col] = df[col].fillna("Missing").astype(str)
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            numerical_cols.append(col)

    return df, feature_cols, categorical_cols, numerical_cols


# 2. Apply our clean pipeline
df_clean, columns, categorical, numerical = \
    convert_dtypes_and_classify(df_master,
        force_categorical=[
            #'INDEX_OF_MULTIPLE_DEPRAVATION_DECILE',
            'MAIN_SPECIALTY_CODE',
            'ETNICITY_DES',
            'GENDER_DESC'])

# Designate numerical features to report as Medians [IQR] (nonnormal) vs Means (normal)
# For clinical data, almost all continuous metrics/counts are best represented as nonnormal [IQR]
nonnormal = numerical

# 3. Instantiate TableOne cleanly
mytable = TableOne(
    data=df_clean,
    columns=columns,
    categorical=categorical,
    nonnormal=nonnormal,
    groupby="esbl_label",
    overall=True
)


# 4. Output results and export
output_path = DATA_DIR / "table1_demographics.csv"
mytable.to_csv(output_path)

print(f"\nTable 1 successfully saved to: {output_path.resolve()}")



Table 1 successfully saved to: /app/data/synthetic/1000p/processed/table1_demographics.csv


In [19]:
# Show
print(mytable.tabulate(tablefmt="github"))

|                                                           |                                      |   Missing | Overall                | Negative               | Positive            |
|-----------------------------------------------------------|--------------------------------------|-----------|------------------------|------------------------|---------------------|
| n                                                         |                                      |           | 1000                   | 850                    | 150                 |
| AGE_AT_ADMISSION, median [Q1,Q3]                          |                                      |         0 | 54.0 [35.0,76.0]       | 54.0 [35.0,76.0]       | 59.0 [36.0,75.8]    |
| MAIN_SPECIALTY_CODE_DESC, n (%)                           | cardiology                           |           | 238 (23.8)             | 200 (23.5)             | 38 (25.3)           |
|                                                           | gastroenterol

In [9]:
# --------------------------------------------------------------------
# How to compute scores if needed
# --------------------------------------------------------------------
# .. note: The computation of scores is also included in 01_vectorized.ipynb.

# Libraries
from pathlib import Path
import pandas as pd

# Local Libraries
from icare_risk.clinphen.scores.gavaghan import compute_gavaghan_score
from icare_risk.clinphen.scores.holmgren import compute_holmgren_score
from icare_risk.clinphen.scores.charlson import compute_charlson_score
from icare_risk.clinphen.scores.jones import compute_jones_score
from icare_risk.clinphen.scores.tumbarello import compute_tumbarello_score
from icare_risk.clinphen.scores.kim import compute_kim_score

# 1. Load your master dataset
DATA_DIR = Path("/app/data/synthetic/10000p/processed/")
filename = '03_derived_analysis_dataset.parquet'
df_master = pd.read_parquet(DATA_DIR / filename)

# 2. compute scores
df_master['charlson_score'] = compute_charlson_score(df_master, verbose=0)
df_master['gavaghan_score'] = compute_gavaghan_score(df_master, verbose=0)
df_master['holmgren_score'] = compute_holmgren_score(df_master, verbose=0)
df_master['jones_score'] = compute_jones_score(df_master, verbose=0)
df_master['tumbarello_score'] = compute_tumbarello_score(df_master, verbose=0)
df_master['kim_score'] = compute_kim_score(df_master, verbose=0)

# Show
score_cols = [
    'charlson_score', 'gavaghan_score', 'holmgren_score', 
    'jones_score', 'tumbarello_score', 'kim_score'
]

df_scores_dist = (
    df_master[score_cols]
    .apply(pd.Series.value_counts)
    .sort_index()     
    .fillna(0)        
    .astype(int)     
)
display(df_scores_dist)

,charlson_score,gavaghan_score,holmgren_score,jones_score,tumbarello_score,kim_score
0.0,2405,9993,10000,9791,9377,9377
1.0,2327,7,0,0,4,590
2.0,1378,0,0,209,586,3
3.0,1241,0,0,0,3,0
4.0,1723,0,0,0,28,0
5.0,816,0,0,0,0,28
6.0,64,0,0,0,2,2
7.0,26,0,0,0,0,0
8.0,6,0,0,0,0,0
9.0,5,0,0,0,0,0
